# 02 · 手刻 Q-learning:一張表學會走迷宮

最經典、也最好懂的 RL 演算法。核心是一張 **Q 表**:`Q[狀態, 動作]` = 「在這個狀態做這個動作,預期能拿到多少長期獎勵」。學會這張表,策略就是「每一步挑 Q 值最大的動作」。

我們用 **FrozenLake**(在結冰湖面從起點走到終點、別掉進冰洞)來練。先用不打滑版本,讓規律單純。

## 1. 安裝與建立環境

In [ ]:
%pip install -q -U "gymnasium[classic-control]"

In [ ]:
import numpy as np
import gymnasium as gym

env = gym.make("FrozenLake-v1", is_slippery=False)   # 先不打滑,規律最單純
n_states = env.observation_space.n                   # 16 格
n_actions = env.action_space.n                       # 4 個方向
print(f"{n_states} 個狀態 × {n_actions} 個動作")

## 2. Q 表 + Q-learning 更新公式

Q 表初始全 0。每走一步,用 **Bellman 更新**把「實際拿到的獎勵 + 未來最好的預期」慢慢灌進這格:

`Q[s,a] ← Q[s,a] + α · ( r + γ·max Q[s',·] − Q[s,a] )`

In [ ]:
Q = np.zeros((n_states, n_actions))
alpha, gamma = 0.8, 0.95          # 學習率、折扣因子(未來獎勵打幾折)
epsilon = 1.0                     # 探索率:一開始全靠亂試
episodes = 2000

for ep in range(episodes):
    s, _ = env.reset()
    done = False
    while not done:
        # epsilon-greedy:有時亂試(探索),有時照目前最好的走(利用)
        if np.random.rand() < epsilon:
            a = env.action_space.sample()
        else:
            a = int(np.argmax(Q[s]))
        s2, r, terminated, truncated, _ = env.step(a)
        done = terminated or truncated
        # Bellman 更新:把 r + 未來最好預期,灌進 Q[s,a]
        Q[s, a] += alpha * (r + gamma * np.max(Q[s2]) - Q[s, a])
        s = s2
    epsilon = max(0.05, epsilon * 0.999)   # 隨著學會,慢慢少亂試
print("訓練完成。epsilon 收斂到", round(epsilon, 3))

## 3. 把學到的策略畫出來

每一格挑 Q 值最大的方向,看 agent 學到怎麼走。

In [ ]:
arrows = ["←", "↓", "→", "↑"]      # FrozenLake 的動作編號對應方向
policy = np.array([arrows[int(np.argmax(Q[s]))] for s in range(n_states)])
print("學到的策略(4×4):")
print(policy.reshape(4, 4))

## 4. 驗收:用學到的策略跑 100 回合,看成功率

In [ ]:
wins = 0
for _ in range(100):
    s, _ = env.reset()
    done = False
    while not done:
        s, r, terminated, truncated, _ = env.step(int(np.argmax(Q[s])))
        done = terminated or truncated
    wins += int(r == 1.0)
print(f"成功抵達終點:{wins}/100 回合")   # 不打滑版本,學成後應該接近 100

## 小結

- **Q-learning = 學一張 `Q[狀態,動作]` 表**,策略就是挑 Q 值最大的動作。
- 三個關鍵:**Bellman 更新**、**折扣因子 γ**、**epsilon-greedy** 平衡探索與利用。
- 不打滑版本能學到接近 100% 成功。把 `is_slippery=True` 打開會難很多——歡迎自己試。

下一課:狀態一多,表就爆了。我們用**神經網路**取代這張表——手刻一個迷你 DQN。